# KazQAD pilot: does the morphological complexity of a Kazakh question affect the answer?

Run the cells **top to bottom**. What each step does:

1. Download KazQAD (public GitHub, CC BY-SA) and load the Kazakh reading-comprehension **test** file.
2. Pick **40 questions**: 20 with the lowest and 20 with the highest *complexity proxy*, with question length and context length restricted so they stay comparable.
3. Build one identical prompt per question and ask **two models** (API, or fill the CSV by hand).
4. **Automatic checks** (code): does the answer contain the reference answer, is it Kazakh, is it a sentence, is it just copied from the text.
5. Export an Excel file for **human scoring** (blind: model and condition are hidden) with the rubric inside.
6. After you fill it in: **analysis** (group means, Mann-Whitney U, Spearman).

**Important honesty notes**
- The complexity measure here is a *crude proxy* (mean word length of the question), not a validated morphological scale. Say this in the report.
- Scores from code are labelled `auto`; scores from a person are labelled `human`. Never mix them up in the report.
- Questions in the two groups are *different* questions (not pairs), so the design is **matched groups**, not paired items.

In [ ]:
# 1. Setup and data download
import os, subprocess, sys
if not os.path.exists("KazQAD"):
    subprocess.run(["git", "clone", "-q", "https://github.com/IS2AI/KazQAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "openai", "google-genai", "openpyxl", "scipy"], check=False)
print("python", sys.version.split()[0])

In [ ]:
# 2. Settings (change here, nowhere else)
CFG = dict(
    N_PER_GROUP = 20,          # 20 low + 20 high = 40 questions
    WORD_RANGE = (5, 7),       # keep only questions with 5-7 words (controls length)
    MAX_CONTEXT_WORDS = 120,   # keep only short contexts (controls length, saves cost)
    SEED = 42,
    DATA_PATH = "KazQAD/data/reading-comprehension/kazqad-reading-comprehension-v1.0-kk-test.jsonl",
)

# Fill in the EXACT model names from each provider's documentation. Add or remove models as needed.
# provider: "openai", "gemini", "groq" or "manual" (manual = you paste answers into the CSV yourself)
MODELS = {
    "A": {"provider": "manual", "model": "GPT (version shown in the chat window)"},   # you paste answers by hand
    "B": {"provider": "gemini", "model": "PUT_EXACT_MODEL_NAME_HERE"},                # API, two keys
    "C": {"provider": "groq",   "model": "PUT_EXACT_MODEL_NAME_HERE"},                # API, Llama via Groq
}

# Prompt instruction in Kazakh. Ask a native speaker to confirm the wording before the real run.
INSTRUCTION = "Мәтін негізінде сұраққа қазақ тілінде бір толық сөйлеммен жауап беріңіз."

In [ ]:
# 3. Load data, compute complexity proxy, select 40 questions
import re, math, json
import pandas as pd

df = pd.read_json(CFG["DATA_PATH"], lines=True)
print("raw rows:", len(df), "| columns:", list(df.columns))

def clean_context(t):
    # KazQAD contexts contain wiki markup like ==Heading==; remove the = signs only.
    return re.sub(r"\s+", " ", re.sub(r"=+", " ", t)).strip()

def words(t):
    return re.findall(r"[^\W\d_]+", t, flags=re.UNICODE)

df["context_clean"] = df["context"].map(clean_context)
df["q_words"] = df["question"].map(lambda t: len(words(t)))
df["ctx_words"] = df["context_clean"].map(lambda t: len(words(t)))
# Complexity proxy: mean letters per word in the question (agglutinative suffixes make words longer).
df["complexity_proxy"] = df["question"].map(lambda t: sum(len(w) for w in words(t)) / max(1, len(words(t))))
df["ref_answers"] = df["answers"].map(lambda a: list(a["text"]))

lo, hi = CFG["WORD_RANGE"]
pool = df[df.q_words.between(lo, hi) & (df.ctx_words <= CFG["MAX_CONTEXT_WORDS"])]
pool = pool.drop_duplicates("question").reset_index(drop=True)
pool = pool[pool.ref_answers.map(len) > 0]
print("questions after length filters:", len(pool))

n = CFG["N_PER_GROUP"]
assert len(pool) >= 2 * n, "Not enough questions after filtering; widen WORD_RANGE or MAX_CONTEXT_WORDS."
pool = pool.sort_values(["complexity_proxy", "id"]).reset_index(drop=True)
low = pool.head(n).assign(condition="low")
high = pool.tail(n).assign(condition="high")
sample = pd.concat([low, high], ignore_index=True)
sample["qid"] = [f"Q{i+1:02d}" for i in range(len(sample))]

# Diagnostics: the groups should differ in the proxy but NOT in length.
print(sample.groupby("condition")[["complexity_proxy", "q_words", "ctx_words"]].mean().round(2))
gap = abs(low.q_words.mean() - high.q_words.mean())
if gap > 0.5:
    print("WARNING: question length differs between groups by", round(gap, 2), "words; narrow WORD_RANGE.")
print()
print("LOW example :", low.iloc[0].question)
print("HIGH example:", high.iloc[0].question)

In [ ]:
# 4. Build prompts and save the questions file
def make_prompt(r):
    return f"Мәтін: {r.context_clean}\nСұрақ: {r.question}\n{INSTRUCTION}"

sample["prompt"] = sample.apply(make_prompt, axis=1)
sample["ref_answer_str"] = sample.ref_answers.map(lambda a: " | ".join(a))
cols = ["qid", "id", "condition", "complexity_proxy", "q_words", "ctx_words", "question", "ref_answer_str", "prompt"]
sample[cols].to_csv("kazqad_selected_questions.csv", index=False, encoding="utf-8-sig")
print("saved kazqad_selected_questions.csv (", len(sample), "questions )")
print()
print(sample.prompt.iloc[0])

In [ ]:
# 4b. Key check: run BEFORE cell 5. It only lists models, it does NOT generate text,
# so it should not use your 20 daily generate-content requests.
import os

def _get(name):
    if os.environ.get(name):
        return os.environ[name]
    from google.colab import userdata
    return userdata.get(name)

def _mask(k):
    return f"{k.strip()[:4]}... (length {len(k.strip())})"

print("=== Gemini keys ===")
from google import genai
seen = {}
gem_model = MODELS["B"]["model"]
for name in ["GEMINI_API_KEY", "GEMINI_API_KEY_2", "GEMINI_API_KEY_3"]:
    try:
        k = _get(name)
    except Exception as e:
        print(name, "-> NOT FOUND in Secrets (or Notebook access is off):", type(e).__name__)
        continue
    problems = []
    if k != k.strip() or any(ch in k for ch in ""' 
	"):
        problems.append("contains spaces / quotes / newline: delete the secret and paste the key again")
    if not k.strip().startswith("AIza"):
        problems.append("does not start with AIza: probably not a Gemini key")
    if k in seen:
        problems.append("same value as " + seen[k] + ": the second key must come from another account or project")
    seen[k] = name
    print(name, _mask(k), "| format problems:", problems if problems else "none")
    try:
        c = genai.Client(api_key=k.strip())
        names = [m.name.replace("models/", "") for m in c.models.list()]
        ok = gem_model in names
        print("   key WORKS. Models visible:", len(names), "| your model", repr(gem_model), "->",
              "available" if ok else "NOT in the list (check the name in the settings cell)")
    except Exception as e:
        print("   key FAILED:", type(e).__name__, str(e)[:120])

print()
print("=== Groq key ===")
from openai import OpenAI
try:
    k = _get("GROQ_API_KEY")
    if k != k.strip() or any(ch in k for ch in ""' 
	"):
        print("GROQ_API_KEY contains spaces / quotes / newline: paste it again")
    if not k.strip().startswith("gsk_"):
        print("GROQ_API_KEY does not start with gsk_: probably not a Groq key")
    c = OpenAI(api_key=k.strip(), base_url="https://api.groq.com/openai/v1")
    ids = [m.id for m in c.models.list()]
    gm = MODELS["C"]["model"]
    print("GROQ_API_KEY", _mask(k), "-> key WORKS. Models visible:", len(ids))
    if gm in ids:
        print("   your model", repr(gm), "-> available")
    else:
        print("   your model", repr(gm), "-> NOT in the list. Llama models available:", [i for i in ids if "llama" in i.lower()])
except Exception as e:
    print("Groq check FAILED:", type(e).__name__, str(e)[:120])

In [ ]:
# 5. Fill kazqad_responses.csv: GPT from your Excel, Gemini (2 keys) and Groq via API. Safe to re-run.
import time, datetime
RESP_FILE = "kazqad_responses.csv"
MANUAL_FILE = "manual_filled.xlsx"                           # your hand-filled GPT answers (model_key A)
GEMINI_KEY_NAMES = ["GEMINI_API_KEY", "GEMINI_API_KEY_2", "GEMINI_API_KEY_3"]   # Colab Secrets names
GEMINI_PER_KEY = 20                                          # questions 1-20 -> key 1, 21-40 -> key 2; key 3 takes over if a key runs out
OTHER_KEY_NAMES = {"openai": "OPENAI_API_KEY", "groq": "GROQ_API_KEY"}
PAUSE_SECONDS = 6                                            # pause between requests (per-minute limits)

def get_key(name):
    # Keys live in Colab: left panel -> key icon (Secrets) -> add the name and the value, switch "Notebook access" on.
    if os.environ.get(name):
        return os.environ[name]
    from google.colab import userdata
    return userdata.get(name)

def ask(provider, model, prompt, key_name):
    if provider in ("openai", "groq"):
        from openai import OpenAI
        kw = {"base_url": "https://api.groq.com/openai/v1"} if provider == "groq" else {}
        client = OpenAI(api_key=get_key(key_name), **kw)
        try:
            r = client.chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}], temperature=0)
        except Exception as e:
            if "temperature" in str(e).lower():     # some models do not accept temperature
                r = client.chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}])
            else:
                raise
        return r.choices[0].message.content
    if provider == "gemini":
        from google import genai
        from google.genai import types
        client = genai.Client(api_key=get_key(key_name))
        r = client.models.generate_content(model=model, contents=prompt, config=types.GenerateContentConfig(temperature=0))
        return r.text
    raise ValueError("unknown provider " + provider)

# --- load what is already saved, drop empty answers (they will be asked again)
done = pd.read_csv(RESP_FILE, encoding="utf-8-sig") if os.path.exists(RESP_FILE) else pd.DataFrame(
    columns=["qid", "model_key", "provider", "model", "timestamp_utc", "response"])
n_before = len(done)
done = done[done["response"].notna() & (done["response"].astype(str).str.strip() != "")]
if len(done) < n_before:
    print("removed", n_before - len(done), "empty responses from the saved file; they will be requested again")

# --- import the hand-filled GPT table (checks that it matches the questions selected now)
if os.path.exists(MANUAL_FILE):
    mm = pd.read_excel(MANUAL_FILE)
    assert mm.response.notna().all(), f"{MANUAL_FILE}: some responses are empty"
    assert set(mm.qid) == set(sample.qid), f"{MANUAL_FILE}: question ids differ from the current selection"
    chk = mm.merge(sample[["qid", "prompt"]], on="qid", suffixes=("_xlsx", "_now"))
    assert (chk.prompt_xlsx.str.strip() == chk.prompt_now.str.strip()).all(),         f"{MANUAL_FILE}: prompts differ from the current selection. Check CFG and re-run cells 3-4."
    mm = mm.drop(columns=["prompt"], errors="ignore")
    done = pd.concat([done[~done.model_key.isin(mm.model_key.unique())], mm], ignore_index=True)
    print("imported", len(mm), "hand-filled answers for model", sorted(mm.model_key.unique()))
else:
    print("NOTE:", MANUAL_FILE, "not found; upload it to use your hand-filled GPT answers")
rows = done.to_dict("records")
pd.DataFrame(rows).to_csv(RESP_FILE, index=False, encoding="utf-8-sig")
have = set(zip(done.qid, done.model_key))

# --- API models
gem_forced = 0   # moves to the next key when a key runs out of daily quota earlier than expected
for key, m in MODELS.items():
    if m["provider"] == "manual":
        continue
    assert "PUT_EXACT" not in m["model"], f"Set the exact model name for model {key} in the settings cell."
    quota_hit = False
    for i, r in enumerate(sample.itertuples()):
        if (r.qid, key) in have:
            continue
        text = None
        slot = 0
        for attempt in range(6):
            if m["provider"] == "gemini":
                slot = min(max(i // GEMINI_PER_KEY, gem_forced), len(GEMINI_KEY_NAMES) - 1)
                key_name = GEMINI_KEY_NAMES[slot]
            else:
                key_name = OTHER_KEY_NAMES[m["provider"]]
            try:
                text = ask(m["provider"], m["model"], r.prompt, key_name)
                break
            except Exception as e:
                msg = str(e)
                print("retry", r.qid, key, type(e).__name__, msg[:100])
                if "PerDay" in msg or "per day" in msg.lower() or "insufficient_quota" in msg:
                    if m["provider"] == "gemini" and slot + 1 < len(GEMINI_KEY_NAMES):
                        gem_forced = slot + 1
                        print(">>> Daily quota over for", key_name, "- switching to", GEMINI_KEY_NAMES[gem_forced])
                        continue
                    print(">>> Daily quota / balance finished for model", key, "(", m["provider"], "). Moving on; run this cell again later.")
                    quota_hit = True
                    break
                if "404" in msg or "403" in msg or "NOT_FOUND" in msg or "PERMISSION" in msg or msg.startswith("401") or "API key not valid" in msg:
                    print("Waiting will not help: check the model name and the key.")
                    break
                time.sleep(10 * (attempt + 1))
        if quota_hit:
            break
        if not text:
            print("FAILED, skipped (will be retried on the next run):", r.qid, key)
            continue
        rows.append(dict(qid=r.qid, model_key=key, provider=m["provider"], model=m["model"],
                         timestamp_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
                         response=text, api_key_slot=(slot + 1 if m["provider"] == "gemini" else "")))
        pd.DataFrame(rows).to_csv(RESP_FILE, index=False, encoding="utf-8-sig")
        time.sleep(PAUSE_SECONDS)

print("responses saved:", len(rows), "->", RESP_FILE)
got = {(x["qid"], x["model_key"]) for x in rows}
for k in MODELS:
    q = [x for x in sample.qid if (x, k) not in got]
    print("model", k, "->", f"all {len(sample)} answers present" if not q else f"missing {len(q)}: {q[:8]}{'...' if len(q) > 8 else ''}")

# MANUAL OPTION: for provider "manual" create the rows yourself in kazqad_responses.csv with the same columns
# (qid, model_key, provider, model, timestamp_utc, response), paste the answer from the chat window,
# and write the visible model name and date in model / timestamp_utc.

In [ ]:
# 6. Automatic checks (code). These are labelled *_auto and are NOT human judgements.
import unicodedata
resp = pd.read_csv(RESP_FILE, encoding="utf-8-sig")
resp["response"] = resp["response"].fillna("").astype(str)
data = resp.merge(sample[["qid", "condition", "complexity_proxy", "question", "ref_answers", "ref_answer_str", "context_clean"]], on="qid", how="left")

def norm(t):
    t = unicodedata.normalize("NFC", t).lower().replace("ё", "е")
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", t)).strip()

def stem(tok):
    if len(tok) < 4:
        return None                     # too short: needs exact match
    return tok[: max(4, math.ceil(0.7 * len(tok)))]

def meaning_auto(response, refs):
    # 2 = a reference answer (normalised) appears as a whole phrase in the response
    # 1 = every word of a reference answer is matched by a response word with the same stem (different suffix)
    # 0 = otherwise
    r = " " + norm(response) + " "
    rt = norm(response).split()
    best = 0
    for ref in refs:
        n_ref = norm(ref)
        if not n_ref:
            continue
        if (" " + n_ref + " ") in r:
            return 2
        toks = n_ref.split()
        ok = True
        for t in toks:
            s = stem(t)
            hit = any(w == t for w in rt) if s is None else any(w.startswith(s) for w in rt)
            ok = ok and hit
        if ok:
            best = max(best, 1)
    return best

def cyrillic_share(t):
    letters = [c for c in t if c.isalpha()]
    return sum("CYRILLIC" in unicodedata.name(c, "") for c in letters) / max(1, len(letters))

data["meaning_auto"] = [meaning_auto(r, refs) for r, refs in zip(data.response, data.ref_answers)]
data["resp_words"] = data.response.map(lambda t: len(words(t)))
data["lang_ok_auto"] = data.response.map(lambda t: int(cyrillic_share(t) >= 0.9))
data["sentence_auto"] = data.resp_words.map(lambda n: int(n >= 3))
data["copied_auto"] = [int(len(norm(r)) > 0 and norm(r) in norm(c)) for r, c in zip(data.response, data.context_clean)]
data["empty_auto"] = (data.resp_words == 0).astype(int)

print(data.groupby(["model_key", "condition"])[["meaning_auto", "lang_ok_auto", "sentence_auto", "copied_auto", "resp_words"]].mean().round(2).to_string())
data.to_csv("kazqad_auto_results.csv", index=False, encoding="utf-8-sig")

In [ ]:
# 7. Export the Excel file for human scoring (blind) + rubric
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation

blind = data.sample(frac=1, random_state=CFG["SEED"]).reset_index(drop=True)
blind["blind_id"] = [f"R{i+1:03d}" for i in range(len(blind))]

wb = Workbook()
# ---- Rubric sheet
ws = wb.active; ws.title = "Rubric"
rubric = [
 ("Criterion", "Score", "Descriptor", "Who scores"),
 ("Meaning (meaning_human)", 2, "The response states the correct answer (same as the reference answer, even if reworded or with a different suffix) and contains nothing that contradicts it.", "Human"),
 ("Meaning (meaning_human)", 1, "The answer is partly correct, vague or incomplete, OR it is correct but accompanied by a wrong or contradicting statement.", "Human"),
 ("Meaning (meaning_human)", 0, "The answer is wrong or missing, or the response does not address the question.", "Human"),
 ("Grammar (grammar_human)", 2, "No evident errors in case, possessive, person/number agreement, verb forms, vowel harmony or word order. Reads naturally to a native speaker.", "Human"),
 ("Grammar (grammar_human)", 1, "One or two minor errors (for example a wrong case suffix or a harmony slip) that do not hinder understanding.", "Human"),
 ("Grammar (grammar_human)", 0, "Three or more errors, or an error that hinders understanding, or the response is mostly not Kazakh or not a sentence.", "Human"),
 ("Grammar is judged on the Kazakh text only", "", "Score grammar even if the answer is wrong, and score meaning even if the grammar is poor. Do not let one score influence the other.", "Human"),
 ("error_type (optional)", "", "Write one or more tags: case / possessive / verb form / agreement / vowel harmony / word order / lexical / language mix / other.", "Human"),
 ("meaning_auto", 2, "The normalised reference answer appears as a whole phrase in the response.", "Code"),
 ("meaning_auto", 1, "Every word of a reference answer is matched by a response word with the same stem (first 70% of letters, at least 4), i.e. a different suffix. Words shorter than 4 letters must match exactly.", "Code"),
 ("meaning_auto", 0, "Otherwise. A 0 can still be correct if reworded: this is why the human score is the primary one.", "Code"),
 ("lang_ok_auto", "1/0", "1 if at least 90% of letters in the response are Cyrillic.", "Code"),
 ("sentence_auto", "1/0", "1 if the response has at least 3 words.", "Code"),
 ("copied_auto", "1/0", "1 if the whole response is an exact substring of the context (copying instead of answering in own words).", "Code"),
 ("Blinding", "", "The Human scoring sheet hides the model name and the question group. Do not open the Key sheet until all scores are entered.", "Procedure"),
]
for r in rubric: ws.append(r)
for c in ws[1]: c.font = Font(bold=True); c.fill = PatternFill("solid", fgColor="D9D9D9")
for row in ws.iter_rows(min_row=1):
    for c in row: c.alignment = Alignment(wrap_text=True, vertical="top")
for col, w in zip("ABCD", (34, 8, 100, 12)): ws.column_dimensions[col].width = w

# ---- Human scoring sheet (blind)
hs = wb.create_sheet("Human scoring")
hs.append(["blind_id", "question", "reference_answer", "response", "meaning_human (0-2)", "grammar_human (0-2)", "error_type", "comment"])
for r in blind.itertuples():
    hs.append([r.blind_id, r.question, r.ref_answer_str, r.response, None, None, None, None])
for c in hs[1]: c.font = Font(bold=True); c.fill = PatternFill("solid", fgColor="D9D9D9")
for col, w in zip("ABCDEFGH", (10, 45, 25, 60, 18, 18, 22, 40)): hs.column_dimensions[col].width = w
for row in hs.iter_rows(min_row=2):
    for c in row: c.alignment = Alignment(wrap_text=True, vertical="top")
dv = DataValidation(type="list", formula1='"0,1,2"', allow_blank=True); hs.add_data_validation(dv)
dv.add(f"E2:F{len(blind)+1}")
hs.freeze_panes = "B2"

# ---- Key sheet (open only after scoring)
ks = wb.create_sheet("Key")
ks.append(["blind_id", "qid", "model_key", "model", "condition", "complexity_proxy", "timestamp_utc", "meaning_auto", "lang_ok_auto", "sentence_auto", "copied_auto", "resp_words"])
for r in blind.itertuples():
    ks.append([r.blind_id, r.qid, r.model_key, r.model, r.condition, round(r.complexity_proxy, 3), r.timestamp_utc,
               r.meaning_auto, r.lang_ok_auto, r.sentence_auto, r.copied_auto, r.resp_words])
for c in ks[1]: c.font = Font(bold=True); c.fill = PatternFill("solid", fgColor="D9D9D9")

wb.save("kazqad_scoring.xlsx")
print("saved kazqad_scoring.xlsx  -> score the 'Human scoring' sheet, save as kazqad_scoring_filled.xlsx and upload it")

In [ ]:
# 8. Analysis AFTER human scoring: upload kazqad_scoring_filled.xlsx to Colab, then run this cell
from scipy import stats
FILLED = "kazqad_scoring_filled.xlsx"
h = pd.read_excel(FILLED, sheet_name="Human scoring").rename(columns={"meaning_human (0-2)": "meaning_human", "grammar_human (0-2)": "grammar_human"})
k = pd.read_excel(FILLED, sheet_name="Key")
res = h.merge(k, on="blind_id")
assert res[["meaning_human", "grammar_human"]].notna().all().all(), "Some human scores are still empty."
assert res[["meaning_human", "grammar_human"]].isin([0, 1, 2]).all().all(), "Scores must be 0, 1 or 2."

summary = res.groupby(["model_key", "condition"]).agg(
    N=("blind_id", "size"),
    meaning_human_mean=("meaning_human", "mean"),
    grammar_human_mean=("grammar_human", "mean"),
    meaning_auto_mean=("meaning_auto", "mean")).round(2)
print(summary.to_string())

print("\nHigh vs low (Mann-Whitney U, two-sided). With n = 20 per group, treat p-values as descriptive.")
for key, g in res.groupby("model_key"):
    for col in ["meaning_human", "grammar_human"]:
        a = g[g.condition == "low"][col]; b = g[g.condition == "high"][col]
        if a.nunique() + b.nunique() <= 2 and a.mean() == b.mean():
            print(key, col, "no variation between groups"); continue
        u = stats.mannwhitneyu(a, b, alternative="two-sided")
        print(f"model {key} | {col}: low mean {a.mean():.2f}, high mean {b.mean():.2f}, U={u.statistic:.1f}, p={u.pvalue:.3f}")

print("\nSpearman correlation between complexity proxy and human score:")
for key, g in res.groupby("model_key"):
    for col in ["meaning_human", "grammar_human"]:
        rho = stats.spearmanr(g.complexity_proxy, g[col])
        print(f"model {key} | {col}: rho={rho.statistic:.2f}, p={rho.pvalue:.3f}")

print("\nHuman vs code agreement on meaning (share of answers where meaning_auto == meaning_human):",
      round((res.meaning_auto == res.meaning_human).mean(), 2))
res.to_csv("kazqad_final_results.csv", index=False, encoding="utf-8-sig")